# Ella2027 — Kaggle LTX video worker
This notebook installs the LTX Diffusers pipeline, loads the distilled 2B checkpoint, then polls your Ella2027 Render queue. Keep the Kaggle session running. A free Kaggle GPU session is limited and may disconnect.

**First:** add Kaggle secret `ELLA_KAGGLE_WORKER_TOKEN` with the same value as Render `KAGGLE_LTX_WORKER_TOKEN`. Do not paste the token into notebook code.

In [ ]:
!nvidia-smi
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg requests

In [ ]:
import os, torch, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient

BASE_URL = 'https://ella2027.onrender.com'
TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
assert TOKEN, 'Kaggle secret ELLA_KAGGLE_WORKER_TOKEN is missing'
health = requests.get(BASE_URL + '/api/kaggle-worker/health', headers={'Authorization': 'Bearer ' + TOKEN}, timeout=45)
print('Render worker health:', health.status_code, health.text)
health.raise_for_status()
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled. In Notebook settings choose GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

MODEL_ID = 'Lightricks/LTX-Video-0.9.8-distilled'
print('Loading model:', MODEL_ID)
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=model_dtype)
pipe.enable_model_cpu_offload()
print('LTX pipeline loaded.')


In [ ]:
# Fetch the queue client from this repository. This code does not contain credentials.
!mkdir -p /kaggle/working/ella_worker
!wget -q -O /kaggle/working/ella_worker/kaggle_worker.py https://raw.githubusercontent.com/WaelNameh84/Ella2027/main/services/kaggle-ltx-worker/kaggle_worker.py
import sys
sys.path.insert(0, '/kaggle/working/ella_worker')
from kaggle_worker import run_kaggle_ltx_worker

def generate_with_ltx(job):
    prompt = str(job.get('prompt') or 'Cinematic realistic scene')
    # Start conservatively on a T4. Increase duration/resolution only after a successful test.
    seconds = min(max(int(job.get('durationSeconds') or 5), 1), 5)
    fps = 8
    frames = seconds * fps + 1
    width, height = 576, 320
    result = pipe(prompt=prompt, width=width, height=height, num_frames=frames,
                  num_inference_steps=8, guidance_scale=1.0)
    video = result.frames[0]
    out = Path('/kaggle/working') / f'ella_{job["id"]}.mp4'
    export_to_video(video, str(out), fps=fps)
    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError('LTX did not produce a valid MP4')
    return out

print('Worker adapter ready. Starting queue polling...')
run_kaggle_ltx_worker(BASE_URL, TOKEN, generate_with_ltx, idle_seconds=15)